In [7]:
import EasyJupyter
import torch
import torch.nn as nn
import torch.nn.functional as F
from model.RoPE import apply_rotary_pos_emb
from typing import Optional

In [8]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig

In [ ]:
def create_causal_doc_mask(cfg: BaseConfig, token_ids: torch.Tensor) -> torch.Tensor:
    """
    Generates a combined causal and document mask.
        - The causal mask prevents tokens from attending to future tokens.
        - The document mask prevents tokens from Doc A from attending to tokens in Doc B. As defined in the Llama 3 paper: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."

    Args:
        token_ids: A mask of shape (batch_size, batch_size, context_len, context_len ).
    """
    batch_size, context_len = token_ids.shape

    # === Create the causal mask ===
    causal_mask = torch.triu(
        torch.full((context_len, context_len), float("-inf"), device=cfg.device),
        diagonal=1,
    )

    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

    # === Create the document mask ===
    # identify the document boundaries
    eod_mask = token_ids == cfg.special_tokens["doc_end_token"]["ID"]

    # Calculate a unique document ID for every token
    #   Tokens at pos N, D1, D1, D1, doc_end_token, D2, D2, ...
    #   doc_ids result: 1, 1, 1, 2, 2, 2, ...
    doc_ids = torch.cumsum(eod_mask.to(torch.long), dim=1) - eod_mask.to(torch.long)

    doc_mask = (doc_ids.unsqueeze(1) != doc_ids.unsqueeze(2))

    # Fill prohibited positions with -inf
    document_blocker = torch.zeros((batch_size, 1, context_len, context_len), device=cfg.device)
    document_blocker = document_blocker.masked_fill(doc_mask, -float("inf"))

    # === Combine the causal and document masks ===
    # The combined_mask will be -inf if a position is in the future OR in a different document.
    combined_mask = causal_mask + document_blocker
    return combined_mask

In [11]:
# @i-c
# TEST
import matplotlib.pyplot as plt
from llama_configs import BaseConfig

cfg = BaseConfig()
context_len = 24
batch_size = 1

# Create mock token IDs with known document boundaries
# The 2 is the doc_end_token
token_ids = torch.tensor([
    [
        [1, 1, 1, 1, 2,  # Doc 1 is 5 tokens long
        1, 1, 1, 1, 1, 1, 1, 1, 1, 2,  # Doc 2 is 10 tokens long
        1, 1, 1, 1, 1, 1, 1, 1, 2]  # Doc 3 is 9 tokens long
    ]], device=cfg.device
).view(batch_size, context_len)


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM


In [12]:
# @i-c
token_ids = token_ids.view(batch_size, context_len)
print(f"Token IDs shape: { token_ids.shape}")

# Test 1: Generate the combined causal and document mask
mask = create_causal_doc_mask(cfg, token_ids)
print(f"Generated mask shape: {mask.shape}")

# test that the standard causal mask is still blocked
assert mask[0, 0, 0, 10] == float("-inf"), "Future tokens are not blocked"

# Test that cross-document attention is blocked
assert mask[0, 0, 10, 1] == float("-inf"), "Attention from Doc 2 to Doc 1 is not blocked"

# Test that internal-document attention is allowed
# Token 10 (Doc 2) attending to Token 6 (Doc 2)
assert mask[0, 0, 10, 6] == 0.0, "Internal-document attention is incorrectly blocked."
print("tests passed")

Token IDs shape: torch.Size([1, 24])
Generated mask shape: torch.Size([1, 1, 24, 24])
tests passed


In [13]:
# @i-c
# Test: Make sure that documents masking properly implementing so that tokens in doc 1 can attend to tokens in doc 2, etc...
doc_boundaries = [(0, 4), (5, 14), (15, 23)]
eod_token_id =cfg.special_tokens["doc_end_token"]["ID"]

print("--- Attention Relationships ---\n")

for q_idx in range(context_len):
    # Determine which document the Query token belongs to
    current_doc = None
    current_doc_id = -1
    for doc_id, (start, end) in enumerate(doc_boundaries):
        if start <= q_idx <= end:
            current_doc = f"Document {doc_id + 1}"
            current_doc_id = doc_id
            break
    
    # Determine if it's the end-of-document token
    q_token_str = f"Token {q_idx}"
    if token_ids[0, q_idx] == eod_token_id:
        q_token_str = f"Token {q_idx} (EOD)"
    
    print(f"{q_token_str} ({current_doc}) can attend to:")

    # Iterate through Key tokens (columns) to see what is allowed to be attended to
    allowed_keys = []
    for k_idx in range(context_len):
        # Key tokens with a mask value of 0.0 are allowed
        if mask[0, 0, q_idx, k_idx] == 0.0:
            # Identify the document of the Key token
            key_doc = None
            for doc_id, (start, end) in enumerate(doc_boundaries):
                if start <= k_idx <= end:
                    key_doc = f"Doc {doc_id + 1}"
                    break
            # Format the key token string
            k_token_str = f"Token {k_idx}"
            if token_ids[0, k_idx] == eod_token_id:
                k_token_str = f"Token {k_idx} (EOD)"
            
            allowed_keys.append(f"{k_token_str} ({key_doc})")
    
    # print the results for this Query token
    if not allowed_keys:
        print("  - [Nothing! Fully masked]")
    else:
        print("  - " + "\n  - ".join(allowed_keys))


--- Attention Relationships ---

Token 0 (Document 1) can attend to:
  - Token 0 (Doc 1)
Token 1 (Document 1) can attend to:
  - Token 0 (Doc 1)
  - Token 1 (Doc 1)
Token 2 (Document 1) can attend to:
  - Token 0 (Doc 1)
  - Token 1 (Doc 1)
  - Token 2 (Doc 1)
Token 3 (Document 1) can attend to:
  - Token 0 (Doc 1)
  - Token 1 (Doc 1)
  - Token 2 (Doc 1)
  - Token 3 (Doc 1)
Token 4 (EOD) (Document 1) can attend to:
  - Token 0 (Doc 1)
  - Token 1 (Doc 1)
  - Token 2 (Doc 1)
  - Token 3 (Doc 1)
  - Token 4 (EOD) (Doc 1)
Token 5 (Document 2) can attend to:
  - Token 5 (Doc 2)
Token 6 (Document 2) can attend to:
  - Token 5 (Doc 2)
  - Token 6 (Doc 2)
Token 7 (Document 2) can attend to:
  - Token 5 (Doc 2)
  - Token 6 (Doc 2)
  - Token 7 (Doc 2)
Token 8 (Document 2) can attend to:
  - Token 5 (Doc 2)
  - Token 6 (Doc 2)
  - Token 7 (Doc 2)
  - Token 8 (Doc 2)
Token 9 (Document 2) can attend to:
  - Token 5 (Doc 2)
  - Token 6 (Doc 2)
  - Token 7 (Doc 2)
  - Token 8 (Doc 2)
  - Token 9 (Do